# Reliable RAG on RGB: E0 baseline and E1 (similarity filter + abstention)

Pipeline: RGB question + passages -> build context for a condition -> prompt -> small LLM -> score.

- **E0** follows the RGB setting: the model answers from all 5 passages, and is told to reply with a **fixed refusal sentence** when the documents do not contain the answer, and a **fixed error sentence** when they contain factual errors.
- **E1** adds two steps around the same prompt: (1) rank the passages by cosine similarity to the question and keep the top K; (2) abstain when even the best passage is not similar enough to the question.

Conditions:
1. **Noise** (irrelevant evidence): noise rate r = 0.0, 0.4, 0.8
2. **Incomplete** (insufficient evidence): r = 1.0, all passages are negatives; correct behaviour is to abstain
3. **Contradictory**: true passage + counterfactual passage + negatives (from `en_fact.json`)

Runtime: NTU CCDS TC1 cluster (1x V100 32GB), run as a Slurm job from `~/rag_project`.

In [ ]:
# RGB data and Python packages are set up once on the TC1 head node (see setup steps),
# so this cell only checks they are in place.
import os
assert os.path.exists("RGB/data/en_refine.json"), "RGB not found - run: git clone https://github.com/chen700564/RGB.git in ~/rag_project"
print(sorted(os.listdir("RGB/data")))

## Datasets

We use the English portion of the RGB benchmark (Chen et al., 2024). Two files are used:

| File | Questions | Fields | Test case |
|---|---|---|---|
| `en_refine.json` | 300 | `query`, `answer`, `positive`, `negative` | **Noise robustness:** 5 passages with noise rate r ∈ {0.0, 0.4, 0.8} (share of negative passages); the answer is always present.<br>**Negative rejection:** r = 1.0, all 5 passages are negatives, so the model should refuse to answer. |
| `en_fact.json` | 100 | `query`, `answer`, `fakeanswer`, `positive`, `positive_wrong`, `negative` | **Counterfactual robustness:** context includes passages edited to state a false answer (`fakeanswer`); the model should detect the error and avoid the false answer. |

- `positive`: retrieved passages that contain the correct answer.
- `negative`: retrieved passages on a related topic that do not contain the answer.
- `positive_wrong`: copies of positive passages with the correct answer replaced by `fakeanswer`.

`en_refine.json` is the authors' corrected version of `en.json` (mislabelled passages removed, some answers fixed). The Chinese files and `en_int.json` (information integration) are not used, as they fall outside this project's scope.

In [ ]:
import json, textwrap
import pandas as pd

DATA_DIR = "RGB/data"
FILES = {
    "en_refine.json": "Noise robustness (r = 0.0-0.8), negative rejection (r = 1.0)",
    "en_fact.json":   "Counterfactual robustness / contradictory evidence",
}
SNIP = 200   # characters shown per passage

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

def show_passages(label, passages, k=2):
    print(f"  {label} ({len(passages)} passages, showing {min(k, len(passages))}):")
    for p in passages[:k]:
        text = " ".join(p[:SNIP].split()) + ("..." if len(p) > SNIP else "")
        print(textwrap.fill(text, 100, initial_indent="    - ", subsequent_indent="      "))

data = {f: load_jsonl(f"{DATA_DIR}/{f}") for f in FILES}

# 1. Summary table
rows = []
for f, d in data.items():
    row = {"file": f, "used for": FILES[f], "questions": len(d),
           "fields": ", ".join(d[0].keys()),
           "avg positive": round(sum(len(x["positive"]) for x in d) / len(d), 1),
           "avg negative": round(sum(len(x["negative"]) for x in d) / len(d), 1)}
    if "positive_wrong" in d[0]:
        row["avg positive_wrong"] = round(sum(len(x["positive_wrong"]) for x in d) / len(d), 1)
    rows.append(row)
display(pd.DataFrame(rows).fillna("-"))

# 2. One example per file (change EXAMPLE_IDX to see another question)
EXAMPLE_IDX = 0
for f, d in data.items():
    x = d[EXAMPLE_IDX]
    print("=" * 100)
    print(f"{f}   (example id {x['id']})")
    print("=" * 100)
    print(f"  query:  {x['query']}")
    print(f"  answer: {x['answer']}")
    if "fakeanswer" in x:
        print(f"  fakeanswer: {x['fakeanswer']}")
    show_passages("positive", x["positive"])
    if "positive_wrong" in x:
        show_passages("positive_wrong", x["positive_wrong"])
    show_passages("negative", x["negative"])
    print()

In [ ]:
import json, random, re, os
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME  = "Qwen/Qwen2.5-3B-Instruct"  # Local LLM
N_PASSAGES  = 5     # Passages per question
SEED        = 42
N_QUESTIONS = None     # None = all; set e.g. 50 for a quick test run first
BATCH_SIZE  = 8
MAX_NEW_TOKENS = 64
OUT_DIR = os.path.expanduser("~/rag_project/results_ranking")   # E0 + E1 (similarity filter + abstention)
os.makedirs(OUT_DIR, exist_ok=True)



In [ ]:
# Refusal detection.
# An output counts as a refusal if it uses the fixed refusal sentence, or a common refusal phrase
# in the model's own words -- unless it is a HEDGE: refusal wording followed by a guess
# ("not explicitly stated ... however, it is likely X"). Hedges count as answers.
REFUSAL     = "I can not answer the question because of insufficient information in documents."
REFUSAL_KEY = "insufficient information"

REFUSAL_PHRASES = [
    REFUSAL_KEY, "can not answer", "cannot answer", "unable to answer",
    "cannot be answered", "cannot be determined", "cannot determine",
    "does not provide", "do not provide", "doesn't provide",
    "does not contain", "do not contain", "doesn't contain",
    "does not specify", "do not specify", "doesn't specify",
    "does not mention", "do not mention", "not mentioned",
    "no information", "no mention", "none of the provided documents", "none of the documents",
    "not explicitly", "not directly",
]

# words that signal the model goes on to guess an answer anyway
HEDGE_MARKERS = [
    "likely", "probably", "presumably", "possibly", "might", "suggest", "infer", "implied",
    "indicating", "as indicated", "assume", "speculat", "expected to", "would be", "would have", "though",
]


## 1. Load RGB

Each file is JSON Lines. `positive` = passages containing the answer, `negative` = retrieved passages that do not.
`en_fact.json` adds `positive_wrong` (passages edited to state `fakeanswer`).

In [ ]:
def load_jsonl(path, n=None):
    with open(path) as f:
        data = [json.loads(line) for line in f if line.strip()]
    return data[:n] if n else data

def flatten(passages):
    # some RGB files store positives as a list of lists; flatten to a list of strings
    out = []
    for p in passages:
        out.extend(p if isinstance(p, list) else [p])
    return out

main = load_jsonl("RGB/data/en_refine.json", N_QUESTIONS)   #questions for noise robustness & negative rejection
fact = load_jsonl("RGB/data/en_fact.json", N_QUESTIONS)     #counterfactual questions
print(len(main), "main questions |", len(fact), "counterfactual questions")
# print(json.dumps({k: (v[:1] if isinstance(v, list) else v) for k, v in main[0].items()}, indent=2)[:1500])

## 2. Build contexts for each condition

Shuffle passages with a fixed seed so (a) the gold passage position carries no signal and (b) your improved method later sees **exactly the same inputs**. The built contexts are saved to disk for reuse.

In [ ]:
def build_noise_context(item, noise_rate, rng):
    pos, neg = flatten(item["positive"]), flatten(item["negative"])
    n_neg = round(N_PASSAGES * noise_rate)
    n_pos = N_PASSAGES - n_neg
    n_neg = min(n_neg, len(neg))
    n_pos = min(n_pos, len(pos))
    docs = pos[:n_pos] + neg[:n_neg]
    rng.shuffle(docs)
    return docs

def build_contradictory_context(item, rng, n_true=1, n_wrong=1):
    pos   = flatten(item["positive"])
    wrong = flatten(item["positive_wrong"])
    neg   = flatten(item["negative"])
    docs = pos[:n_true] + wrong[:n_wrong]
    docs += neg[: max(0, N_PASSAGES - len(docs))]
    rng.shuffle(docs)
    return docs

def make_examples(condition, noise_rate=None):
    rng = random.Random(SEED)
    examples = []
    if condition == "contradictory":
        for it in fact:
            examples.append({"id": it["id"], "query": it["query"], "answer": it["answer"],
                             "fakeanswer": it.get("fakeanswer"),
                             "docs": build_contradictory_context(it, rng)})
    else:
        for it in main:
            examples.append({"id": it["id"], "query": it["query"], "answer": it["answer"],
                             "docs": build_noise_context(it, noise_rate, rng)})
    return examples

CONDITIONS = {
    "noise_0.0":     make_examples("noise", 0.0),
    "noise_0.4":     make_examples("noise", 0.4),
    "noise_0.8":     make_examples("noise", 0.8),
    "incomplete":    make_examples("noise", 1.0),
    "contradictory": make_examples("contradictory"),
}
for name, ex in CONDITIONS.items():
    with open(f"{OUT_DIR}/inputs_{name}.jsonl", "w") as f:
        for e in ex:
            f.write(json.dumps(e) + "\n")
    print(f"{name:14s} {len(ex)} examples")

## 3. Prompt (E0)

The prompt numbers the passages, adds the question, and gives three instructions: answer if the documents contain the answer; otherwise reply with the fixed refusal sentence; if the documents contain errors, start with the fixed error sentence and give the correct answer. The refusal sentence is the same one that the E1 methods insert when they abstain (`REFUSAL` below).

In [ ]:
SYSTEM = "You are an accurate and reliable assistant that answers questions using only the provided documents."

def format_docs(docs):
    # collapse runs of whitespace left over from web scraping (shorter prompts, faster generation)
    return "\n\n".join(f"[{i+1}] {' '.join(d.split())}" for i, d in enumerate(docs))

def build_prompt(query, docs):
    return (
            f"Documents:\n{format_docs(docs)}\n\n"
            f"Question: {query}\n\n"
            "Instructions:\n"
            "- If the documents contain the answer, reply with a short answer.\n"
            "- If the documents do not contain enough information to answer, reply exactly: "
            "\"I can not answer the question because of insufficient information in documents.\"\n"
            "- If the documents contain conflicting or incorrect information, start your reply with "
            "\"There are factual errors in the provided documents.\" and then give the correct answer."
        )

ex = CONDITIONS["noise_0.4"][0]
print(build_prompt(ex["query"], ex["docs"])[-700:])   # last part: question + instruction

## 4. Load the generator

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"          # required for batched generation with decoder-only models
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16, device_map="auto"   # fp16: V100 has no bf16 support
)
model.eval()
print(torch.cuda.get_device_name(0), "|", round(torch.cuda.memory_allocated() / 1e9, 2), "GB used")

In [ ]:
@torch.no_grad()
def generate(prompts):
    outputs = []
    for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
        batch = prompts[i:i + BATCH_SIZE]
        texts = [tokenizer.apply_chat_template(
                    [{"role": "system", "content": SYSTEM}, {"role": "user", "content": p}],
                    tokenize=False, add_generation_prompt=True) for p in batch]
        enc = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        new_tokens = out[:, enc["input_ids"].shape[1]:]   # strip the prompt
        outputs.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
    return [o.strip() for o in outputs]

## 5. Scoring (RGB rules)

RGB's `answer` can be nested: the outer list holds parts that are **all** required; an inner list holds alternatives where **any one** counts. Matching is case-insensitive substring matching.

- **Refusal:** the fixed refusal sentence, or a common refusal phrase in the model's own words (`REFUSAL_PHRASES`).
- **Hedge:** refusal wording followed by a guess (`HEDGE_MARKERS`, e.g. "not explicitly stated ... however, it is likely X"). Hedges count as **answers**, not refusals.
- Every output is exactly one of **correct** (contains the gold answer), **rejected** (refusal without the gold answer) or **wrong** (any other answer).
- **Noise and contradictory:** accuracy, plus rejection rate (refusing although the answer was in the documents) and wrong-answer rate.
- **Incomplete:** rejection rate.

In [ ]:
def contains_answer(pred, answer):
    pred = pred.lower()
    if isinstance(answer, str):
        answer = [answer]
    for part in answer:
        alts = part if isinstance(part, list) else [part]
        if not any(str(a).lower() in pred for a in alts):
            return False
    return True

def is_hedge(pred):
    p = pred.lower()
    return any(x in p for x in REFUSAL_PHRASES) and any(h in p for h in HEDGE_MARKERS)

def is_rejection(pred):
    p = pred.lower()
    if REFUSAL_KEY in p:                              # the instructed sentence always counts
        return True
    if not any(x in p for x in REFUSAL_PHRASES):      # no refusal wording -> an answer
        return False
    return not is_hedge(pred)                          # refusal wording, but not followed by a guess

def score(name, examples, preds):
    rows = []
    for e, p in zip(examples, preds):
        rows.append({"id": e["id"], "query": e["query"], "pred": p,
                     "correct": contains_answer(p, e["answer"]),
                     "rejected": is_rejection(p),
                     "hedge": is_hedge(p) and REFUSAL_KEY not in p.lower()})
    
    df = pd.DataFrame(rows)
    df["rejected"] = df.rejected & ~df.correct          # a refusal that contains the answer counts as correct
    df["wrong"]    = ~df.correct & ~df.rejected

    if name == "incomplete":
        summary = {"rejection_rate": df.rejected.mean()}
    else:   # noise_* and contradictory
        summary = {"accuracy": df.correct.mean(),
                   "rejection_rate": df.rejected.mean(),
                   "wrong_answer": df.wrong.mean()}
    return df, summary

## 6. Run E0 on all conditions

`run_method` generates, scores and saves predictions for one method into its own folder (`OUT_DIR/<method>/`). E1 reuses it with filtered passages.

If a prediction file already exists with the right number of rows, it is loaded and rescored instead of regenerated, so an interrupted run resumes where it stopped. **Delete the folder to force a rerun** (e.g. after changing a prompt).

In [ ]:
def run_method(method, build_prompt_fn, conditions=None):
    conditions = conditions or CONDITIONS   # E1 passes filtered passages here
    mdir = f"{OUT_DIR}/{method}"
    os.makedirs(mdir, exist_ok=True)
    summaries = {}
    for name, examples in conditions.items():
        print(f"\n=== {method} | {name} ===")
        path = f"{mdir}/preds_{name}.jsonl"
        preds = None
        if os.path.exists(path):
            old = pd.read_json(path, lines=True)
            if len(old) == len(examples):
                preds = old["pred"].fillna("").tolist()
                print("loaded saved predictions")
        if preds is None:
            preds = generate([build_prompt_fn(e["query"], e["docs"]) for e in examples])
        df, summary = score(name, examples, preds)
        df.to_json(path, orient="records", lines=True)
        summaries[name] = summary
        print({k: round(v, 3) for k, v in summary.items()})
    return summaries

BASELINE = "E0_refusal"
E0_summaries = run_method(BASELINE, build_prompt)

In [ ]:
E0_results = pd.DataFrame(E0_summaries).T.round(3)
E0_results.to_csv(f"{OUT_DIR}/{BASELINE}/summary.csv")
E0_results.fillna("")   # blank = metric not used for that condition

## 7. Inspect E0 failures

Look at real outputs before moving on: incomplete questions where the model did not use the refusal sentence, and contradictory questions it got wrong.

In [ ]:
def show_failures(method, k=10):
    mdir = f"{OUT_DIR}/{method}"
    inc = pd.read_json(f"{mdir}/preds_incomplete.jsonl", lines=True)
    print(f"[{method}] Incomplete: did not use the refusal sentence ({(~inc.rejected).sum()} of {len(inc)}):")
    display(inc[~inc.rejected][["query", "pred"]].head(k))

    con = pd.read_json(f"{mdir}/preds_contradictory.jsonl", lines=True)
    print(f"[{method}] Contradictory: wrong answer ({(~con.correct).sum()} of {len(con)}):")
    display(con[~con.correct][["query", "pred"]].head(k))

show_failures(BASELINE)


# E1: Similarity filter + abstention

E1 keeps the E0 prompt and model, and changes **which passages the model sees** and **when it is allowed to answer**.

1. **Filter.** Embed the question and each passage with a bi-encoder (`all-mpnet-base-v2`, the dense retriever RGB used to collect the passages) and compute cosine similarity. Keep the top `TOP_K` passages, most similar first, and generate with the E0 prompt.
2. **Abstain.** If the highest passage similarity is below a threshold τ, output the fixed refusal sentence instead of the model's answer.

The result of both steps together is reported as **`E1_filter_abstain`**.

**Calibration / test split.** τ is chosen on 30% of the questions (calibration) and all results are reported on the other 70% (test). The split is by question id and is the same for every method, including E0.

**Choosing τ.** τ maximises the *balanced score* on the calibration split, ½(mean noise accuracy + incomplete rejection rate). Contradictory questions are not used for choosing τ; they are reported separately. If several τ values tie, the middle one is used.

**Expected weakness.** The bi-encoder embeds the question and passage separately, so it mostly measures topic. RGB's negatives are on the same topic (e.g. a different year), so they may score as high as the positives. The diagnostics below (recall@K and similarity separation) measure how much this matters.

In [ ]:
from sklearn.metrics import roc_auc_score
from sentence_transformers import SentenceTransformer
import numpy as np

EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"   # RGB's own dense retriever
TOP_K       = 2          # passages kept after filtering
CALIB_FRAC  = 0.3
ANSWERABLE  = ["noise_0.0", "noise_0.4", "noise_0.8"]
POOL        = ANSWERABLE + ["incomplete"]                 # used to choose tau
METHOD_F    = "E1_filter"
METHOD_FA   = "E1_filter_abstain"
for m in [METHOD_F, METHOD_FA]:
    os.makedirs(f"{OUT_DIR}/{m}", exist_ok=True)

def cached(path, n, compute):
    # reuse saved results if they exist and have the right length (lets an interrupted run resume)
    if os.path.exists(path):
        df = pd.read_json(path, lines=True)
        if len(df) == n:
            print("loaded saved results")
            return df.to_dict("records")
    rows = compute()
    pd.DataFrame(rows).to_json(path, orient="records", lines=True)
    return rows

def load_preds(method):
    out = {}
    for name, examples in CONDITIONS.items():
        df = pd.read_json(f"{OUT_DIR}/{method}/preds_{name}.jsonl", lines=True)
        assert len(df) == len(examples), f"{method} predictions for {name} missing or incomplete"
        out[name] = df["pred"].fillna("").astype(str).tolist()
    return out

E0_PREDS = load_preds(BASELINE)

# calibration / test split by question id (same split for every method)
_rng = random.Random(SEED)
_main_ids = sorted({e["id"] for e in CONDITIONS["incomplete"]})
_fact_ids = sorted({e["id"] for e in CONDITIONS["contradictory"]})
CALIB_IDS = {
    "main": set(_rng.sample(_main_ids, int(CALIB_FRAC * len(_main_ids)))),
    "fact": set(_rng.sample(_fact_ids, int(CALIB_FRAC * len(_fact_ids)))),
}
def in_split(name, qid, split):
    calib = CALIB_IDS["fact" if name == "contradictory" else "main"]
    return (qid in calib) == (split == "calib")

print(f"calibration: {len(CALIB_IDS['main'])} main + {len(CALIB_IDS['fact'])} counterfactual questions; rest = test")

## E1 step 1: passage similarity and filtering

One embedding pass per question (the question plus its 5 passages); no LLM calls. Similarities are saved, so a rerun loads them.

In [ ]:
embedder = SentenceTransformer(EMBED_MODEL, device="cuda")

def passage_sims(examples):
    rows = []
    for e in tqdm(examples):
        q = embedder.encode([e["query"]], normalize_embeddings=True)[0]
        d = embedder.encode(e["docs"], normalize_embeddings=True)
        rows.append({"sims": (d @ q).tolist()})       # cosine similarity (embeddings are unit length)
    return rows

SIMS = {}
for name, examples in CONDITIONS.items():
    print(f"=== similarity | {name} ===")
    SIMS[name] = [r["sims"] for r in cached(f"{OUT_DIR}/{METHOD_F}/sims_{name}.jsonl", len(examples),
                                            lambda: passage_sims(examples))]

MAX_SIM = {name: [max(s) for s in SIMS[name]] for name in CONDITIONS}   # abstention score

def top_k(docs, sims, k=TOP_K):
    order = np.argsort(sims)[::-1][:k]                 # most similar first
    return [docs[i] for i in order]

FILTERED = {name: [{**e, "docs": top_k(e["docs"], s)} for e, s in zip(examples, SIMS[name])]
            for name, examples in CONDITIONS.items()}

### Filter diagnostics

RGB labels which passages contain the answer, so the filter can be checked before generation:
- **recall@K** (noise): share of questions where at least one positive passage survives the filter. If this is low, the filter is throwing the answer away. At noise 0.0 every passage is positive, so it is 1 by construction.
- **contradictory:** how often the true passage, the fake passage, or both survive. Both passages are on topic, so the filter is not expected to separate them.
- **similarity separation:** mean top similarity per condition, and AUROC of the top similarity for answerable (noise) vs unanswerable (incomplete) questions on the test split. This is how well the abstention score can work at all (0.5 = no separation).

In [ ]:
POS  = {it["id"]: set(flatten(it["positive"]))       for it in main}
TRUE = {it["id"]: set(flatten(it["positive"]))       for it in fact}
FAKE = {it["id"]: set(flatten(it["positive_wrong"])) for it in fact}

def kept(name, label):
    return [any(d in label[e["id"]] for d in f["docs"]) for e, f in zip(CONDITIONS[name], FILTERED[name])]

diag = {name: {f"recall@{TOP_K}": np.mean(kept(name, POS))} for name in ANSWERABLE}
t, f = np.array(kept("contradictory", TRUE)), np.array(kept("contradictory", FAKE))
diag["contradictory"] = {"true kept": t.mean(), "fake kept": f.mean(), "both kept": (t & f).mean()}
for name in CONDITIONS:
    diag.setdefault(name, {})["mean top similarity"] = np.mean(MAX_SIM[name])
display(pd.DataFrame(diag).T.round(3).fillna(""))

y, s = [], []
for name in POOL:
    for e, m in zip(CONDITIONS[name], MAX_SIM[name]):
        if in_split(name, e["id"], "test"):
            y.append(name != "incomplete"); s.append(m)
print(f"AUROC of top similarity, noise vs incomplete (test split): {roc_auc_score(y, s):.3f}")

### Generate with the filtered passages

Same prompt and model as E0; only the passages change. This is the one generation pass E1 needs. The outputs are saved in `E1_filter/` and are an intermediate step: abstention is applied to them below.

In [ ]:
_ = run_method(METHOD_F, build_prompt, FILTERED)   # intermediate: filtered outputs before abstention
E1F_PREDS = load_preds(METHOD_F)

## E1 step 2: abstention on top similarity

Keep the filtered output if the top passage similarity ≥ τ, otherwise output the fixed refusal sentence. The result is `E1_filter_abstain`.

In [ ]:
def apply_threshold(preds, scores, tau):
    return {name: [p if s >= tau else REFUSAL for p, s in zip(preds[name], scores[name])]
            for name in CONDITIONS}

def evaluate(final_preds, split):
    out = {}
    for name, examples in CONDITIONS.items():
        idx = [i for i, e in enumerate(examples) if in_split(name, e["id"], split)]
        _, out[name] = score(name, [examples[i] for i in idx], [final_preds[name][i] for i in idx])
    return out

def balanced(summ):
    noise_acc = np.mean([summ[n]["accuracy"] for n in ANSWERABLE])
    return (noise_acc + summ["incomplete"]["rejection_rate"]) / 2

def choose_threshold(preds, scores):
    calib = sorted({s for name in POOL for e, s in zip(CONDITIONS[name], scores[name])
                    if in_split(name, e["id"], "calib")})
    candidates = [-np.inf] + calib
    vals = [balanced(evaluate(apply_threshold(preds, scores, t), "calib")) for t in tqdm(candidates)]
    best = max(vals)
    tied = [t for t, v in zip(candidates, vals) if v >= best - 1e-12]
    return float(tied[len(tied) // 2])        # middle of the tied range

TAU = choose_threshold(E1F_PREDS, MAX_SIM)
E1FA_PREDS = apply_threshold(E1F_PREDS, MAX_SIM, TAU)
print(f"tau = {TAU:.4f}")

# save the final outputs (all questions, with split label) for error analysis
for name, examples in CONDITIONS.items():
    df, _ = score(name, examples, E1FA_PREDS[name])
    df["top_sim"] = MAX_SIM[name]
    df["split"] = ["calib" if in_split(name, e["id"], "calib") else "test" for e in examples]
    df.to_json(f"{OUT_DIR}/{METHOD_FA}/preds_{name}.jsonl", orient="records", lines=True)

## E1 results (test split)

Both rows are scored on the same 70% test split.

- **acc**: output contains the gold answer
- **rej noise_0.8 / rej contradictory**: refused although the answer was in the documents (lower is better)
- **wrong contradictory**: answered, but not with the gold answer
- **rej incomplete**: refused when no passage had the answer (higher is better)
- **balanced**: the τ-selection criterion, ½(mean noise accuracy + incomplete rejection rate)

In [ ]:
def summary_row(summ):
    return {
        "acc noise_0.0":       summ["noise_0.0"]["accuracy"],
        "acc noise_0.4":       summ["noise_0.4"]["accuracy"],
        "acc noise_0.8":       summ["noise_0.8"]["accuracy"],
        "rej noise_0.8":       summ["noise_0.8"]["rejection_rate"],
        "rej incomplete":      summ["incomplete"]["rejection_rate"],
        "acc contradictory":   summ["contradictory"]["accuracy"],
        "rej contradictory":   summ["contradictory"]["rejection_rate"],
        "wrong contradictory": summ["contradictory"]["wrong_answer"],
        "balanced":            balanced(summ),
    }

final_table = pd.DataFrame({
    BASELINE:  summary_row(evaluate(E0_PREDS, "test")),
    METHOD_FA: summary_row(evaluate(E1FA_PREDS, "test")),
}).T.round(3)
final_table["tau"] = [np.nan, round(TAU, 3)]
final_table.to_csv(f"{OUT_DIR}/E1_comparison.csv")
final_table.fillna("")

## Inspect E1 failures

- noise 0.8 questions where the filter dropped every positive passage (the answer never reached the model)
- incomplete questions that were still answered after filtering and abstention

In [ ]:
def show_e1_failures(k=10):
    lost = [(e["query"], p) for e, hit, p in zip(CONDITIONS["noise_0.8"], kept("noise_0.8", POS), E1F_PREDS["noise_0.8"])
            if not hit]
    print(f"noise_0.8: positive passage dropped by the filter ({len(lost)} of {len(CONDITIONS['noise_0.8'])}):")
    display(pd.DataFrame(lost[:k], columns=["query", "pred"]))

    inc = pd.read_json(f"{OUT_DIR}/{METHOD_FA}/preds_incomplete.jsonl", lines=True)
    inc = inc[(inc.split == "test") & ~inc.rejected]
    print(f"incomplete (test): still answered after abstention ({len(inc)}):")
    display(inc[["query", "top_sim", "pred"]].head(k))

show_e1_failures()